In [ ]:
%matplotlib widget

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from scipy.optimize import root

from ipywidgets import (
    HTML,
    HTMLMath,
    VBox,
    HBox,
    Layout
)

from IPython.display import display

# ============================================================
# CHEBYSHEV RATIONAL MINIMAX APPROXIMATION
#
# Example:
#
# f(x) = exp(x)
# interval [-1,1]
# w(x) = 1
#
# Rational approximation:
#
# R_11(x) = (c0 + c1*x)/(1 + d1*x)
#
# For m = 1, k = 1:
#
# N = m+k = 2
# L = N+2 = 4 alternating extrema
#
# IMPORTANT:
# The coefficients of the rational approximation are NOT
# entered beforehand. They are computed numerically from
# the equioscillation conditions.
# ============================================================

plt.ioff()

# ============================================================
# CLASSIC JUPYTER + JUPYTERLAB / NOTEBOOK 7 / BINDER
# ============================================================

display(HTML("""
<style>

.container {
    width: 98% !important;
    max-width: none !important;
}

.output_area,
.output_subarea {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.output_scroll {
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
    box-shadow: none !important;
}

.jp-Cell-outputWrapper,
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.widget-output,
.jupyter-widgets-output-area,
.widget-box {
    max-width: none !important;
    height: auto !important;
    max-height: none !important;
    overflow: visible !important;
}

.jupyter-matplotlib,
.jupyter-matplotlib-figure {
    overflow: visible !important;
    resize: none !important;
}

.jupyter-matplotlib::-webkit-resizer,
.jupyter-matplotlib-figure::-webkit-resizer {
    display: none !important;
}

.rat-title {
    font-family:Arial, sans-serif;
    font-size:20px;
    font-weight:bold;
    color:#6f3fa0;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    width:1280px;
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.48;
    margin-bottom:10px;
">

<div class="rat-title" style="margin-bottom:8px;">
Chebyshev Rational Minimax Approximation
</div>

<div style="margin-bottom:5px;">
This notebook demonstrates the equioscillation principle for the rational
approximation of f(x) = exp(x) on the interval [-1,1].
</div>

<div style="margin-bottom:5px;">
We seek a rational function of type (1,1),
R₁₁(x) = (c₀+c₁x)/(1+d₁x), with weight w(x)=1.
</div>

<div>
Since m+k=2, the nondegenerate minimax approximation must exhibit
four alternating extrema of the error. The coefficients, error level,
and two interior extremal points are determined numerically from the
alternation equations; they are not entered beforehand.
</div>

</div>
""")

# ============================================================
# SYMBOLIC DEFINITIONS
# ============================================================

x = sp.symbols(
    'x',
    real=True
)

c0, c1, d1 = sp.symbols(
    'c_0 c_1 d_1',
    real=True
)

rho = sp.symbols(
    'r',
    real=True
)

f_symbolic = sp.exp(x)

R_symbolic = (
    c0 + c1*x
) / (
    1 + d1*x
)

E_symbolic = sp.simplify(
    f_symbolic - R_symbolic
)

dE_symbolic = sp.diff(
    E_symbolic,
    x
)

# ============================================================
# NUMERICAL ALTERNATION SYSTEM
#
# Alternating points:
#
# x1 = -1
# x2 = unknown interior point
# x3 = unknown interior point
# x4 = +1
#
# E(-1) = +r
# E(x2) = -r
# E(x3) = +r
# E(+1) = -r
#
# E'(x2) = 0
# E'(x3) = 0
# ============================================================

def alternation_system(values):

    C0, C1, D1, RHO, x2, x3 = values

    def rational(xx):

        return (
            C0 + C1*xx
        ) / (
            1.0 + D1*xx
        )

    def rational_derivative(xx):

        return (
            C1 - D1*C0
        ) / (
            1.0 + D1*xx
        )**2

    def error(xx):

        return (
            np.exp(xx)
            -
            rational(xx)
        )

    def error_derivative(xx):

        return (
            np.exp(xx)
            -
            rational_derivative(xx)
        )

    return np.array(
        [
            error(-1.0) - RHO,
            error(x2) + RHO,
            error(x3) - RHO,
            error(+1.0) + RHO,
            error_derivative(x2),
            error_derivative(x3)
        ],
        dtype=float
    )

# ============================================================
# SOLVE THE NONLINEAR SYSTEM
# ============================================================

initial_guess = np.array(
    [
        1.0,     # c0
        1.0,     # c1
        0.20,    # d1
        0.02,    # r
        -0.25,   # x2
        0.70     # x3
    ],
    dtype=float
)

solution = root(
    alternation_system,
    initial_guess
)

if not solution.success:

    raise RuntimeError(
        "The alternation system did not converge."
    )

C0, C1, D1, RHO, X2, X3 = solution.x

# ============================================================
# ORDER INTERIOR EXTREMAL POINTS IF NECESSARY
# ============================================================

if X2 > X3:

    X2, X3 = X3, X2

# ============================================================
# NUMERICAL FUNCTIONS
# ============================================================

def f_numeric(xx):

    return np.exp(xx)


def R_numeric(xx):

    return (
        C0 + C1*xx
    ) / (
        1.0 + D1*xx
    )


def E_numeric(xx):

    return (
        f_numeric(xx)
        -
        R_numeric(xx)
    )

# ============================================================
# DENSE GRID
# ============================================================

x_values = np.linspace(
    -1.0,
    1.0,
    2000
)

f_values = f_numeric(
    x_values
)

R_values = R_numeric(
    x_values
)

E_values = E_numeric(
    x_values
)

# ============================================================
# ALTERNATING POINTS
# ============================================================

alternation_x = np.array(
    [
        -1.0,
        X2,
        X3,
        1.0
    ]
)

alternation_E = E_numeric(
    alternation_x
)

# ============================================================
# NUMERICAL CHECKS
# ============================================================

max_error_grid = np.max(
    np.abs(
        E_values
    )
)

min_denominator = np.min(
    np.abs(
        1.0 + D1*x_values
    )
)

# ============================================================
# MATHEMATICAL RESULTS
#
# IMPORTANT:
# No \qquad commands are used.
# ============================================================

general_form_math = HTMLMath(
    value=(
        r'\('
        r'R_{11}(x)'
        r'='
        r'\dfrac{c_0+c_1x}{1+d_1x}'
        r'\)'
    )
)

computed_form_math = HTMLMath(
    value=(
        r'\('
        r'R_{11}^{*}(x)'
        r'='
        r'\dfrac{'
        +
        f'{C0:.8f}'
        +
        '+'
        +
        f'{C1:.8f}'
        +
        r'x}{1'
        +
        f'{D1:+.8f}'
        +
        r'x}'
        r'\)'
    )
)

error_level_math = HTMLMath(
    value=(
        r'\('
        r'r_{11}^{*}'
        r'\approx'
        +
        f'{abs(RHO):.8f}'
        +
        r'\)'
    )
)

points_math = HTMLMath(
    value=(
        r'\('
        r'x_1=-1,\;'
        r'x_2='
        +
        f'{X2:.8f}'
        +
        r',\;'
        r'x_3='
        +
        f'{X3:.8f}'
        +
        r',\;'
        r'x_4=1'
        r'\)'
    )
)

alternation_math = HTMLMath(
    value=(
        r'\('
        r'E(x_1)\approx+r,\;'
        r'E(x_2)\approx-r,\;'
        r'E(x_3)\approx+r,\;'
        r'E(x_4)\approx-r'
        r'\)'
    )
)

# ============================================================
# RESULTS PANEL
# ============================================================

results_panel = VBox(
    [
        HTML("""
        <div class="rat-title" style="margin-bottom:8px;">
            Computed Minimax Approximation
        </div>
        """),

        general_form_math,
        computed_form_math,
        error_level_math,
        points_math,
        alternation_math,

        HTML(
            f"""
            <div style="
                font-family:Arial;
                font-size:13px;
                line-height:1.5;
                margin-top:5px;
            ">

            Maximum error on dense grid:
            <b>{max_error_grid:.8f}</b>

            <br>

            Minimum |Q₁(x)| on [-1,1]:
            <b>{min_denominator:.8f}</b>

            </div>
            """
        )
    ],
    layout=Layout(
        width='1280px',
        padding='10px 14px',
        border='1px solid #d2c2df',
        overflow='visible'
    )
)

# ============================================================
# FIGURE 1
# FUNCTION AND APPROXIMATION
# ============================================================

fig_fun, ax_fun = plt.subplots(
    figsize=(6.4, 4.7)
)

fig_fun.canvas.header_visible = False
fig_fun.canvas.footer_visible = False
fig_fun.canvas.toolbar_visible = False

fig_fun.canvas.layout = Layout(
    width='640px',
    height='470px',
    overflow='visible'
)

ax_fun.set_title(
    'Function and Rational Approximation',
    fontsize=14,
    fontweight='bold',
    color='#6f3fa0'
)

ax_fun.set_xlabel(
    'x'
)

ax_fun.set_ylabel(
    'Function value'
)

ax_fun.set_xlim(
    -1.0,
    1.0
)

ax_fun.grid(
    True,
    linestyle=':',
    alpha=0.40
)

ax_fun.plot(
    x_values,
    f_values,
    linewidth=2.0,
    label='f(x) = exp(x)'
)

ax_fun.plot(
    x_values,
    R_values,
    linestyle='--',
    linewidth=2.0,
    label='R₁₁*(x)'
)

# ============================================================
# LEGEND BELOW FIGURE 1
# HORIZONTAL ARRANGEMENT
# ============================================================

ax_fun.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.17),
    ncol=2,
    fontsize=9,
    frameon=True
)

fig_fun.subplots_adjust(
    left=0.12,
    right=0.97,
    top=0.90,
    bottom=0.25
)

# ============================================================
# FIGURE 2
# ERROR AND EQUIOSCILLATION
# ============================================================

fig_err, ax_err = plt.subplots(
    figsize=(6.4, 4.7)
)

fig_err.canvas.header_visible = False
fig_err.canvas.footer_visible = False
fig_err.canvas.toolbar_visible = False

fig_err.canvas.layout = Layout(
    width='640px',
    height='470px',
    overflow='visible'
)

ax_err.set_title(
    'Approximation Error and Equioscillation',
    fontsize=14,
    fontweight='bold',
    color='#0b3d91'
)

ax_err.set_xlabel(
    'x'
)

ax_err.set_ylabel(
    'E(x) = f(x) − R₁₁*(x)'
)

ax_err.set_xlim(
    -1.0,
    1.0
)

error_limit = (
    1.20
    *
    max_error_grid
)

ax_err.set_ylim(
    -error_limit,
    error_limit
)

ax_err.grid(
    True,
    linestyle=':',
    alpha=0.40
)

ax_err.axhline(
    0.0,
    linewidth=1.0
)

ax_err.axhline(
    abs(RHO),
    linestyle='--',
    linewidth=1.2,
    label='+r'
)

ax_err.axhline(
    -abs(RHO),
    linestyle='--',
    linewidth=1.2,
    label='−r'
)

ax_err.plot(
    x_values,
    E_values,
    linewidth=2.0,
    label='E(x)'
)

ax_err.plot(
    alternation_x,
    alternation_E,
    linestyle='None',
    marker='o',
    markersize=7,
    label='Alternating extrema'
)

# ============================================================
# LEGEND BELOW FIGURE 2
# ALL FOUR ENTRIES IN ONE HORIZONTAL ROW
# ============================================================

ax_err.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.17),
    ncol=4,
    fontsize=9,
    frameon=True
)

fig_err.subplots_adjust(
    left=0.13,
    right=0.97,
    top=0.90,
    bottom=0.25
)

# ============================================================
# FIGURES ROW
# ============================================================

figures_row = HBox(
    [
        fig_fun.canvas,
        fig_err.canvas
    ],
    layout=Layout(
        width='1300px',
        gap='15px',
        align_items='flex-start',
        overflow='visible'
    )
)

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        documentation,
        results_panel,
        figures_row
    ],
    layout=Layout(
        width='1320px',
        gap='10px',
        overflow='visible'
    )
)

display(
    main_layout
)